# FedTrace — 06: Combined Accountability Record

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/06_combined.ipynb)

**Runtime:** CPU only
**Estimated time:** ~5 min
**Input checkpoints:** `data/raw/02_*.jsonl` (notebook 02), `data/raw/05_grant_outlays.jsonl` (notebook 05)
**Publishes to GitHub:** `data/raw/06_awards.jsonl`, `data/outputs/06_combined.json`, `data/findings/06_combined.md`

**Context:** Notebooks 02–05 produced three-number records separately for contracts and grants. This notebook assembles them into a single per-award accountability record with a unified schema, then produces cross-cutting analysis across all cancelled awards.

**Driving questions:**
1. Across all cancelled awards (contracts + grants), what is the combined ceiling / obligated / outlays record?
2. How does the obligation-to-outlay gap differ between contracts and grants — how much committed money was still unpaid at cancellation?
3. What fraction of awards were near-zero obligation at cancellation (ceiling never reached)?

**Per-award schema:** `award_id`, `award_type`, `agency`, `recipient`, `description`, `period_start`, `period_end`, `ceiling`, `obligated`, `outlays`, `ceiling_gap`, `obligation_outlay_gap`, `ceiling_gap_pct`, `outlay_rate`, `three_number_complete`, `doge_savings`, `outlay_source`.

**Current findings:** `data/findings/06_combined.md`.

## Setup

Run cells 1–3 at the start of every session.

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('06_combined', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['raw_dir'].mkdir(parents=True, exist_ok=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
(PATHS['data_root'] / 'findings').mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
# ── CELL 3: Imports & helpers ──────────────────────────────────────────────────────
import json
import pandas as pd
from pathlib import Path


def load_jsonl(path: Path, label: str = '') -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(
            f'{label or path.name} not found at {path}. '
            'Run the prerequisite notebook first.'
        )
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def _f(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None


def _safe_pct(a, b):
    return a / (b or 1) * 100


def _gap(a, b):
    if a is None or b is None:
        return None
    return round(a - b, 2)


def _gap_pct(ceiling, obligated):
    if ceiling is None or obligated is None or ceiling <= 0:
        return None
    return round((ceiling - obligated) / ceiling * 100, 2)


def _outlay_rate(outlays, obligated):
    if outlays is None or obligated is None or obligated <= 0:
        return None
    return round(outlays / obligated, 4)

In [ ]:
# ── CELL 4: Load input checkpoints ───────────────────────────────────────────────
raw = PATHS['raw_dir']

contracts_path   = raw / '02_contracts_matched.jsonl'
idv_amounts_path = raw / '02_idv_amounts.jsonl'
grants_path      = raw / '02_grants.jsonl'
outlays_path     = raw / '05_grant_outlays.jsonl'

awards_path      = raw / '06_awards.jsonl'
output_json_path = PATHS['outputs_dir'] / '06_combined.json'
findings_md_path = PATHS['data_root'] / 'findings' / '06_combined.md'

contracts_recs   = load_jsonl(contracts_path,   'contracts matched (nb02)')
idv_amounts_recs = load_jsonl(idv_amounts_path, 'IDV amounts (nb02)')
grants_recs      = load_jsonl(grants_path,      'grants (nb02)')
outlays_recs     = load_jsonl(outlays_path,     'grant outlays (nb05)')

print('Checkpoints loaded:')
print(f'  contracts matched:  {len(contracts_recs):,}')
print(f'  IDV amounts:        {len(idv_amounts_recs):,}')
print(f'  grants:             {len(grants_recs):,}')
print(f'  grant outlays:      {len(outlays_recs):,}')

## 2. Contract Records

Reconstruct the per-contract three-number record from notebook 02 checkpoints.
Logic mirrors notebook 03; no API calls.

In [ ]:
# ── Contract assembly ────────────────────────────────────────────────────────────────────────
matched_df  = pd.DataFrame(contracts_recs)
idv_amts_df = pd.DataFrame(idv_amounts_recs)

# Join IDV child amounts onto IDV rows
if not idv_amts_df.empty and 'generated_internal_id' in matched_df.columns:
    idv_amts_df = idv_amts_df.rename(columns={
        'child_award_total_obligation': '_idv_obligated',
        'child_award_total_outlay':     '_idv_outlays',
    })
    idv_join = idv_amts_df[['generated_internal_id', '_idv_obligated', '_idv_outlays']].drop_duplicates(
        subset='generated_internal_id', keep='last'
    )
    matched_df = matched_df.merge(idv_join, on='generated_internal_id', how='left')
else:
    matched_df['_idv_obligated'] = None
    matched_df['_idv_outlays']   = None

# Standardise columns
if 'piid' not in matched_df.columns and 'Award ID' in matched_df.columns:
    matched_df = matched_df.rename(columns={'Award ID': 'piid'})

col_map = {
    'Recipient Name':                         'recipient_name',
    'Period of Performance Start Date':       'period_start',
    'Period of Performance Current End Date': 'period_end',
}
matched_df = matched_df.rename(columns={k: v for k, v in col_map.items() if k in matched_df.columns})

# Resolve ceiling / obligated / outlays
matched_df['ceiling']   = matched_df['value'].apply(_f)
matched_df['obligated'] = matched_df.apply(
    lambda r: _f(r.get('_idv_obligated')) if r.get('record_type') == 'idv' else _f(r.get('Award Amount')),
    axis=1,
)
matched_df['outlays'] = matched_df.apply(
    lambda r: _f(r.get('_idv_outlays')) if r.get('record_type') == 'idv' else _f(r.get('Total Outlays')),
    axis=1,
)

contract_records = []
for _, row in matched_df.iterrows():
    c = row['ceiling']
    o = row['obligated']
    u = row['outlays']
    contract_records.append({
        'award_id':              row.get('piid'),
        'award_type':            row.get('record_type', 'contract'),
        'agency':                row.get('agency') or row.get('awarding_agency'),
        'recipient':             row.get('recipient_name'),
        'description':           row.get('description'),
        'period_start':          row.get('period_start'),
        'period_end':            row.get('period_end'),
        'ceiling':               c,
        'obligated':             o,
        'outlays':               u,
        'ceiling_gap':           _gap(c, o),
        'obligation_outlay_gap': _gap(o, u),
        'ceiling_gap_pct':       _gap_pct(c, o),
        'outlay_rate':           _outlay_rate(u, o),
        'three_number_complete': all(v is not None for v in [c, o, u]),
        'doge_savings':          _f(row.get('savings')),
        'outlay_source':         'usaspending_api',
    })

three = sum(1 for r in contract_records if r['three_number_complete'])
print(f'Contract records: {len(contract_records):,}')
print(f'  three-number complete: {three:,} ({_safe_pct(three, len(contract_records)):.1f}%)')

## 3. Grant Records

Join notebook 02 grant data (ceiling, obligated, metadata) with notebook 05 outlay data.

In [ ]:
# ── Grant assembly ─────────────────────────────────────────────────────────────────────
# Build outlay lookup: award_id -> record with highest total_outlays (handles resume duplicates)
outlay_map = {}
for rec in outlays_recs:
    aid = rec.get('award_id')
    if not aid:
        continue
    new_val = _f(rec.get('total_outlays'))
    prev    = outlay_map.get(aid)
    if prev is None or (new_val is not None and new_val > (_f(prev.get('total_outlays')) or 0)):
        outlay_map[aid] = rec

grant_records = []
for row in grants_recs:
    aid       = row.get('award_id')
    c         = _f(row.get('value'))
    o         = _f(row.get('total_obligation'))
    odata     = outlay_map.get(aid, {})
    u         = _f(odata.get('total_outlays'))
    grant_records.append({
        'award_id':              aid,
        'award_type':            'grant',
        'agency':                row.get('awarding_agency') or row.get('agency'),
        'recipient':             row.get('recipient_name') or row.get('recipient'),
        'description':           row.get('description'),
        'period_start':          row.get('period_start'),
        'period_end':            row.get('period_end'),
        'ceiling':               c,
        'obligated':             o,
        'outlays':               u,
        'ceiling_gap':           _gap(c, o),
        'obligation_outlay_gap': _gap(o, u),
        'ceiling_gap_pct':       _gap_pct(c, o),
        'outlay_rate':           _outlay_rate(u, o),
        'three_number_complete': all(v is not None for v in [c, o, u]),
        'doge_savings':          _f(row.get('savings')),
        'outlay_source':         'bulk_archive' if u is not None else 'none',
    })

three = sum(1 for r in grant_records if r['three_number_complete'])
print(f'Grant records: {len(grant_records):,}')
print(f'  three-number complete: {three:,} ({_safe_pct(three, len(grant_records)):.1f}%)')

## 4. Combine and Analyse

In [ ]:
# ── Combine ─────────────────────────────────────────────────────────────────────────────────
all_records = contract_records + grant_records
df = pd.DataFrame(all_records)

for col in ['ceiling', 'obligated', 'outlays', 'ceiling_gap', 'obligation_outlay_gap', 'ceiling_gap_pct', 'outlay_rate']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

n_total    = len(df)
n_three    = int(df['three_number_complete'].sum())
n_contract = int(df['award_type'].isin(['contract', 'idv']).sum())
n_grant    = int((df['award_type'] == 'grant').sum())

print(f'Combined: {n_total:,} awards ({n_contract:,} contracts/IDVs, {n_grant:,} grants)')
print(f'Three-number complete: {n_three:,} ({_safe_pct(n_three, n_total):.1f}%)')
print()

def _agg(col):
    s = df[col]
    return round(float(s.sum()), 0) if s.notna().any() else None

agg_all = {
    'total_ceiling':               _agg('ceiling'),
    'total_obligated':             _agg('obligated'),
    'total_outlays':               _agg('outlays'),
    'total_obligation_outlay_gap': round(float(df.loc[df['obligation_outlay_gap'].notna(), 'obligation_outlay_gap'].sum()), 0),
}

print('Aggregate (all awards):')
for k, v in agg_all.items():
    print(f'  {k}: ${(v or 0):,.0f}')

In [ ]:
# ── By award type ───────────────────────────────────────────────────────────────────────────
agg_by_type = {}
for atype in ['contract', 'idv', 'grant']:
    sub = df[df['award_type'] == atype]
    if sub.empty:
        continue
    three = int(sub['three_number_complete'].sum())
    obl_gap = sub['obligation_outlay_gap'].dropna()
    agg_by_type[atype] = {
        'n':                           len(sub),
        'all_three_numbers':           three,
        'all_three_numbers_pct':       round(_safe_pct(three, len(sub)), 1),
        'total_ceiling':               round(float(sub['ceiling'].sum()), 0),
        'total_obligated':             round(float(sub['obligated'].sum()), 0),
        'total_outlays':               round(float(sub['outlays'].sum()), 0) if sub['outlays'].notna().any() else None,
        'total_obligation_outlay_gap': round(float(obl_gap.sum()), 0) if not obl_gap.empty else None,
    }

for atype, stats in agg_by_type.items():
    print(f'{atype.upper()}  (n={stats["n"]:,}, three-number: {stats["all_three_numbers_pct"]}%)')
    print(f'  ceiling:    ${stats["total_ceiling"]:,.0f}')
    print(f'  obligated:  ${stats["total_obligated"]:,.0f}')
    out = stats.get("total_outlays")
    print(f'  outlays:    ${(out or 0):,.0f}' + ('' if out is not None else '  (partial coverage)'))
    gap = stats.get("total_obligation_outlay_gap")
    print(f'  obl-outlay gap: ${(gap or 0):,.0f}')
    print()

In [ ]:
# ── Obligation-to-outlay gap distribution ─────────────────────────────────────────────
# The obligation-to-outlay gap is the most direct measure of financial disruption:
# money legally committed but not yet disbursed when the award was cancelled.

valid = df[df['obligated'].notna() & df['outlays'].notna() & (df['obligated'] > 0)].copy()
valid['outlay_rate_f'] = valid['outlays'] / valid['obligated']

outlay_stats = {}
if not valid.empty:
    outlay_stats = {
        'n':                           len(valid),
        'median_outlay_rate':          round(float(valid['outlay_rate_f'].median()), 3),
        'mean_outlay_rate':            round(float(valid['outlay_rate_f'].mean()),   3),
        'pct_gt_80pct_disbursed':      round(_safe_pct((valid['outlay_rate_f'] > 0.8).sum(), len(valid)), 1),
        'pct_lt_20pct_disbursed':      round(_safe_pct((valid['outlay_rate_f'] < 0.2).sum(), len(valid)), 1),
        'total_obligation_outlay_gap': round(float(valid['obligation_outlay_gap'].sum()), 0),
    }
    print('Obligation-to-outlay gap (awards with both obligated and outlays present):')
    for k, v in outlay_stats.items():
        if isinstance(v, (int, float)) and abs(v) > 1000:
            print(f'  {k}: ${v:,.0f}')
        else:
            print(f'  {k}: {v}')

# Near-zero obligation
nz = df[df['ceiling'].notna() & df['obligated'].notna() & (df['ceiling'] > 0)].copy()
nz['obl_rate'] = nz['obligated'] / nz['ceiling']
n_near_zero = int((nz['obl_rate'] < 0.01).sum())
print(f'\nNear-zero obligation (<1% of ceiling): {n_near_zero:,} ({_safe_pct(n_near_zero, len(nz)):.1f}%)')

## 5. Publish

In [ ]:
# ── Write per-award JSONL ──────────────────────────────────────────────────────────────────
with open(awards_path, 'w') as f:
    for rec in all_records:
        f.write(json.dumps(rec) + '\n')

print(f'Written: {awards_path.name}')
print(f'  {len(all_records):,} records, {awards_path.stat().st_size / 1e6:.1f} MB')

In [ ]:
# ── Aggregate JSON ───────────────────────────────────────────────────────────────────────
output = {
    'source_notebooks': ['02_fetch', '05_grant_outlays_bulk'],
    'all_awards': {
        'total':                 n_total,
        'contracts_idvs':        n_contract,
        'grants':                n_grant,
        'all_three_numbers':     n_three,
        'all_three_numbers_pct': round(_safe_pct(n_three, n_total), 1),
        'aggregate':             agg_all,
        'outlay_rate':           outlay_stats,
        'near_zero_obligation_at_cancellation': n_near_zero,
    },
    'by_award_type': agg_by_type,
    'methodology_notes': [
        'ceiling = DOGE value field',
        'obligated = USASpending Award Amount (contracts), child_award_total_obligation (IDVs), total_obligation (grants)',
        'outlays = USASpending Total Outlays (contracts/IDVs), total_outlayed_amount_for_overall_award bulk archive (grants)',
        'obligation_outlay_gap = obligated - outlays: money committed but not disbursed at cancellation',
        'ceiling_gap = ceiling - obligated: unexercised contractual capacity',
        'DOGE savings = ceiling - current obligations; does not net termination costs',
        'Grant outlay coverage: 67.9% (32.1% have no outlay data in FY2020-FY2026 archive files)',
    ],
}

output_json_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

In [ ]:
# ── Findings MD ─────────────────────────────────────────────────────────────────────────
def _fmt(v):
    return f'${(v or 0):,.0f}' if v is not None else 'N/A'

c_stats = agg_by_type.get('contract', {})
i_stats = agg_by_type.get('idv', {})
g_stats = agg_by_type.get('grant', {})

c_n  = c_stats.get('n', 0) + i_stats.get('n', 0)
c_c  = (c_stats.get('total_ceiling', 0) or 0)   + (i_stats.get('total_ceiling', 0) or 0)
c_o  = (c_stats.get('total_obligated', 0) or 0) + (i_stats.get('total_obligated', 0) or 0)
c_u  = (c_stats.get('total_outlays', 0) or 0)   + (i_stats.get('total_outlays', 0) or 0)

lines = [
    '# Findings \u2014 06: Combined Accountability Record',
    '',
    f'*Input: {n_total:,} matched awards ({n_contract:,} contracts/IDVs, {n_grant:,} grants).*  ',
    f'*Methodology: `notebooks/06_combined.ipynb`*',
    '',
    '## Confirmed',
    '',
    '**Combined three-number record \u2014 all cancelled awards:**',
    '',
    '| | Ceiling | Obligated | Outlays |',
    '|---|---|---|---|',
    f'| Contracts / IDVs ({c_n:,}) | ${c_c:,.0f} | ${c_o:,.0f} | ${c_u:,.0f} |',
    f'| Grants ({g_stats.get("n", 0):,}) | {_fmt(g_stats.get("total_ceiling"))} | {_fmt(g_stats.get("total_obligated"))} | {_fmt(g_stats.get("total_outlays"))} |',
    f'| **All ({n_total:,})** | **{_fmt(agg_all.get("total_ceiling"))}** | **{_fmt(agg_all.get("total_obligated"))}** | **{_fmt(agg_all.get("total_outlays"))}** |',
    '',
    f'**Three-number completeness:** {n_three:,}/{n_total:,} ({_safe_pct(n_three, n_total):.1f}%).',
    '',
    '**Obligation-to-outlay gap** (obligated minus outlays \u2014 money committed but not yet disbursed at cancellation):',
    f'- Total: {_fmt(outlay_stats.get("total_obligation_outlay_gap"))} (across {outlay_stats.get("n", 0):,} awards with both fields present)',
    f'- Median outlay rate: {outlay_stats.get("median_outlay_rate", 0):.1%}',
    f'- {outlay_stats.get("pct_gt_80pct_disbursed", 0)}% of awards were >80% disbursed at cancellation',
    f'- {outlay_stats.get("pct_lt_20pct_disbursed", 0)}% of awards were <20% disbursed at cancellation',
    '',
    f'**Near-zero obligation at cancellation** (<1% of ceiling obligated): {n_near_zero:,} awards ({_safe_pct(n_near_zero, n_total):.1f}%).',
    '',
    '**Methodology constraints:**',
    '- Grant outlay coverage is 67.9%. The obligation-to-outlay gap for the remaining 32.1% of grants is null.',
    '- DOGE savings = ceiling minus current obligations; does not net termination costs.',
    '- Ceiling gap reflects unexercised contractual capacity, not recovered funds.',
    '',
    '## Open',
    '',
    '- Linkage path for 22% of DOGE grants with no USASpending link (3,510 records) \u2014 obligated/outlays unavailable.',
    '- GenAI layer: plain-language per-award summary from this record \u2014 notebook 07.',
]

findings_md = '\n'.join(lines)
findings_md_path.write_text(findings_md)
print(findings_md)

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'data/raw/06_awards.jsonl',
        'data/outputs/06_combined.json',
        'data/findings/06_combined.md',
    ],
    message='Combined accountability record \u2014 contracts and grants',
    repo_dir=REPO,
)